Charger la data processed

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# charger la data processed
data_filled = pd.read_csv('../data/processed/chip_chain_processed.csv')
data_daily_returns = pd.read_csv('../data/processed/chip_chain_daily_returns.csv')
data_log_returns = pd.read_csv('../data/processed/chip_chain_log_returns.csv')

## Trouver des paramètres où la série est stationnaire

### Test ADF (Augmented Dickey-Fuller)

Création d'un fonction test ADF :
- On veut trouver un paramètre où la série est stationnaire pour avoir une loi qui ne dépende qui a un comprtement statistique toujours valable

Une série temporelle est stationnaire :
- si la moyenne est constante
- si la variance est constante
- si la covariance dépend seulement du lag (Cov(X_t, X_(t-k))=y(k))

Pour faire ce test au lieu de calculer ces trois conditions, on utilise Dickey-Fuller qui a prouvé que si une série possède une racine unitaire, alors elle se comporte comme une marche aléatoire, ainsi on peut montrer la série est non stationnaire (hypothèse nulle H0)

Soit X_t = a*X_(t-1) + E_t
- si |a| < 1 alors stationnaire
- si a = 1 alors racine unitaire (marche aléatoire)

In [2]:
def test_ADF_finance(data, column, p_value_threshold=0.05):
    from statsmodels.tsa.stattools import adfuller
    
    result = adfuller(data[column].dropna())
    p_value = result[1]
    
    print(f"ADF Statistic: {result[0]}")
    print(f"p-value: {p_value}")
    
    if p_value < p_value_threshold:
        print("The series could be stationary (reject H0)")
    else:
        print("The series is non-stationary (fail to reject H0)")

In [3]:
print("Testing ADF on filled data:")
test_ADF_finance(data_filled, 'NVDA')

print("\nTesting ADF on daily returns:")
test_ADF_finance(data_daily_returns, 'NVDA')

print("\nTesting ADF on log returns:")
test_ADF_finance(data_log_returns, 'NVDA')

Testing ADF on filled data:
ADF Statistic: 1.8007619134972237
p-value: 0.9983505269843606
The series is non-stationary (fail to reject H0)

Testing ADF on daily returns:
ADF Statistic: -21.330874070119627
p-value: 0.0
The series could be stationary (reject H0)

Testing ADF on log returns:
ADF Statistic: -21.293540793093772
p-value: 0.0
The series could be stationary (reject H0)


On voit bien que data seulement filled donc les prix bruts ne sont pas stationnaires, tandis que les rendements simples et log PEUVENT être stationnaires

### Test de KPSS (Kwiatkowski-Phillips-Schmidt-Shin)

Néanmoins, on n'a pas prouvé que les deux types de rendements sont stationnaires, c'est pourquoi on va faire le test de KPSS, où H0 est la série EST constante

In [4]:
def test_KPSS_finance(data, column, p_value_threshold=0.05):
    from statsmodels.tsa.stattools import kpss
    
    result = kpss(data[column].dropna(), regression='c') # le 'c' signifie que nous testons la stationnarité autour d'une constante (niveau stationnaire)
    p_value = result[1]
    
    print(f"KPSS Statistic: {result[0]}")
    print(f"p-value: {p_value}")
    
    if p_value < p_value_threshold:
        print("The series is non-stationary (reject H0)")
    else:
        print("The series is stationary (fail to reject H0)")

In [5]:
print("\nTesting KPSS on filled data:")
test_KPSS_finance(data_filled, 'NVDA')

print("\nTesting KPSS on daily returns:")
test_KPSS_finance(data_daily_returns, 'NVDA')

print("\nTesting KPSS on log returns:")
test_KPSS_finance(data_log_returns, 'NVDA')


Testing KPSS on filled data:
KPSS Statistic: 5.182182076491971
p-value: 0.01
The series is non-stationary (reject H0)

Testing KPSS on daily returns:
KPSS Statistic: 0.23908781555365508
p-value: 0.1
The series is stationary (fail to reject H0)

Testing KPSS on log returns:
KPSS Statistic: 0.16251704292042374
p-value: 0.1
The series is stationary (fail to reject H0)


C:\Users\rolli\AppData\Local\Temp\ipykernel_26348\1127506512.py:4: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result = kpss(data[column].dropna(), regression='c') # le 'c' signifie que nous testons la stationnarité autour d'une constante (niveau stationnaire)
C:\Users\rolli\AppData\Local\Temp\ipykernel_26348\1127506512.py:4: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(data[column].dropna(), regression='c') # le 'c' signifie que nous testons la stationnarité autour d'une constante (niveau stationnaire)
C:\Users\rolli\AppData\Local\Temp\ipykernel_26348\1127506512.py:4: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value retur

Les messages en haut viennent du fait que pour le test KPSS, statsmodels ne calcule pas la p-value mais utilise une table correspondance avec des valeurs entre 0.01 et 0.1. Les prix bruts sont trop petits et les rendements sont trop grands, mais permette quand même de vérifier les deux car 0.05 est entre 0.1 et 0.01

Ainsi on voit bien que les deux types de rendements sont stationnaires

## Trouver des paramètres où la série est autocorrélée

### Test de Ljung-Box

On veut maintenant prouver que le marché est autocorrélé pour pouvoir prédire quelque chose. Car s'il était 100% efficient et aléatoire (cf Eugène Fama notamment sur le côté efficient (on a besoin d'avoir un temps de digération de l'information)), l'intelligence artificelle ne servirait à rien

In [6]:
def test_Ljung_box_finance(data, column, lags=[1, 5, 10], p_value_threshold=0.05):
    from statsmodels.stats.diagnostic import acorr_ljungbox
    
    result = acorr_ljungbox(data[column].dropna(), lags=lags)
    p_values = result["lb_pvalue"]
    
    print(f"Ljung-Box p-values for lags 1 to {lags}: {p_values}")

    if any(p < p_value_threshold for p in p_values):
        print("There is evidence of autocorrelation (reject H0)")
    else:
        print("There is no evidence of autocorrelation (fail to reject H0)")

In [7]:
print("Testing Ljung-Box on filled data:")
test_Ljung_box_finance(data_filled, 'NVDA')

print("\nTesting Ljung-Box on daily returns:")
test_Ljung_box_finance(data_daily_returns, 'NVDA')

print("\nTesting Ljung-Box on log returns:")
test_Ljung_box_finance(data_log_returns, 'NVDA')

Testing Ljung-Box on filled data:
Ljung-Box p-values for lags 1 to [1, 5, 10]: 1     0.0
5     0.0
10    0.0
Name: lb_pvalue, dtype: float64
There is evidence of autocorrelation (reject H0)

Testing Ljung-Box on daily returns:
Ljung-Box p-values for lags 1 to [1, 5, 10]: 1     0.000014
5     0.000127
10    0.000002
Name: lb_pvalue, dtype: float64
There is evidence of autocorrelation (reject H0)

Testing Ljung-Box on log returns:
Ljung-Box p-values for lags 1 to [1, 5, 10]: 1     0.000023
5     0.000132
10    0.000002
Name: lb_pvalue, dtype: float64
There is evidence of autocorrelation (reject H0)


In [8]:
# pour VIX
print("Testing Ljung-Box on VIX filled data:")
test_Ljung_box_finance(data_filled, '^VIX')
print("\nTesting Ljung-Box on VIX daily returns:")
test_Ljung_box_finance(data_daily_returns, '^VIX')
print("\nTesting Ljung-Box on VIX log returns:")
test_Ljung_box_finance(data_log_returns, '^VIX')

Testing Ljung-Box on VIX filled data:
Ljung-Box p-values for lags 1 to [1, 5, 10]: 1     0.0
5     0.0
10    0.0
Name: lb_pvalue, dtype: float64
There is evidence of autocorrelation (reject H0)

Testing Ljung-Box on VIX daily returns:
Ljung-Box p-values for lags 1 to [1, 5, 10]: 1     0.000415
5     0.000565
10    0.000457
Name: lb_pvalue, dtype: float64
There is evidence of autocorrelation (reject H0)

Testing Ljung-Box on VIX log returns:
Ljung-Box p-values for lags 1 to [1, 5, 10]: 1     0.000017
5     0.000005
10    0.000004
Name: lb_pvalue, dtype: float64
There is evidence of autocorrelation (reject H0)


In [9]:
# pour PICK
print("Testing Ljung-Box on PICK filled data:")
test_Ljung_box_finance(data_filled, 'PICK')
print("\nTesting Ljung-Box on PICK daily returns:")
test_Ljung_box_finance(data_daily_returns, 'PICK')
print("\nTesting Ljung-Box on PICK log returns:")
test_Ljung_box_finance(data_log_returns, 'PICK')

Testing Ljung-Box on PICK filled data:
Ljung-Box p-values for lags 1 to [1, 5, 10]: 1     0.0
5     0.0
10    0.0
Name: lb_pvalue, dtype: float64
There is evidence of autocorrelation (reject H0)

Testing Ljung-Box on PICK daily returns:
Ljung-Box p-values for lags 1 to [1, 5, 10]: 1     0.117007
5     0.053251
10    0.000031
Name: lb_pvalue, dtype: float64
There is evidence of autocorrelation (reject H0)

Testing Ljung-Box on PICK log returns:
Ljung-Box p-values for lags 1 to [1, 5, 10]: 1     0.117993
5     0.049204
10    0.000021
Name: lb_pvalue, dtype: float64
There is evidence of autocorrelation (reject H0)


On remarque que toutes les paramètres sont autocorrélées. Néanmoins, on remarque que certaines sont plus fortement autocorrélées que d'autres :
- d'abord les prix bruts ont une plus forte autocorrélation que les rendements (même si pas stationnaire)
- les rendements qui sont auto-corrélés ont une autocorrélation plus forte plus le lag est grand (en tout cas jusqu'à 10)

Les prix bruts sont évidemment beaucoup plus autocorrélés que les rendements car le prix d'aujourd'hui est fortement lié au prix d'hier. Les rendements, en revanche, sont souvent plus proches de l'indépendance, surtout les rendements log

C'est pour cela que la chose la plus importante qui a été trouvé dans ce test est que l'autocorrélation est plus fote avec lag importante

### On stocke tous la donnée pour tous les tickers

In [10]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings
warnings.filterwarnings("ignore") 

def generate_statistical_report(log_returns_df):
    results = []
    numeric_df = log_returns_df.select_dtypes(include=[np.number])
    
    # Les 4 horizons de temps (1 semaine, 2 semaines, 1 mois, 1 trimestre)
    lags_to_test = [5, 10, 21, 63] 
    
    for ticker in numeric_df.columns:
        data = numeric_df[ticker].dropna()
        
        # 1. Test ADF
        adf_stat, adf_p, _, _, _, _ = adfuller(data)
        
        # 2. Test KPSS
        kpss_stat, kpss_p, _, _ = kpss(data, regression='c')
        
        # 3. Test Ljung-Box sur plusieurs Lags
        lb_df = acorr_ljungbox(data, lags=lags_to_test)
        
        # On extrait les p-values pour chaque lag
        p_5 = lb_df.loc[5, 'lb_pvalue']
        p_10 = lb_df.loc[10, 'lb_pvalue']
        p_21 = lb_df.loc[21, 'lb_pvalue']
        p_63 = lb_df.loc[63, 'lb_pvalue']
        
        results.append({
            'Ticker': ticker,
            'Is_Stationary': (adf_p < 0.05) and (kpss_p > 0.05),
            'LB_p_1Week (5)': round(p_5, 4),
            'LB_p_2Weeks (10)': round(p_10, 4),
            'LB_p_1Month (21)': round(p_21, 4),
            'LB_p_1Quarter (63)': round(p_63, 4),
            # On considère qu'il y a de la mémoire si AU MOINS UN des lags est < 0.05
            'Has_Any_Memory': any(p < 0.05 for p in [p_5, p_10, p_21, p_63])
        })
        
    return pd.DataFrame(results)

# Lancement et sauvegarde
report_df = generate_statistical_report(data_log_returns)
print(report_df)
report_df.to_csv('../data/processed/statistical_summary.csv', index=False)
print("\nRapport multi-lags sauvegardé avec succès !")

     Ticker  Is_Stationary  LB_p_1Week (5)  LB_p_2Weeks (10)  \
0   0981.HK           True          0.0912            0.0249   
1      AAPL           True          0.2225            0.0000   
2       AMD           True          0.0019            0.0049   
3      ASML           True          0.0000            0.0000   
4      AVGO           True          0.0000            0.0000   
5       CGW           True          0.0000            0.0000   
6     GOOGL           True          0.0488            0.0000   
7      INTC           True          0.0000            0.0000   
8      MSFT           True          0.0000            0.0000   
9      NVDA           True          0.0001            0.0000   
10     PICK           True          0.0492            0.0000   
11     TSLA           True          0.8491            0.5681   
12      TSM           True          0.0000            0.0000   
13      USO           True          0.0869            0.0507   
14     ^TNX           True          0.00

On remarque tous sont stationnaires et que certains ne sont autocorrélés pour des lags petits USO (pétrole) n'ai pas autocorrélé (normal s'est plus lent que la tech) mais pour des grands lags, il est plus corrélés que les tickers techs et indicateurs (VIX et TNX)

### Ressources à utiliser ppour approfondir le sujet proposées par l'IA

- Le Livre de vulgarisation absolue (Le "Must-Read")
  - "A Random Walk Down Wall Street" (Une marche au hasard à travers la bourse) de Burton G. Malkiel.
Pourquoi le lire : Ce n'est pas de Fama lui-même, mais c'est le livre qui a popularisé sa théorie pour le grand public. Il explique avec beaucoup d'humour pourquoi un singe aux yeux bandés lançant des fléchettes sur les pages financières d'un journal fera un aussi bon portefeuille qu'un expert (si le marché est 100% efficient). C'est un classique mondial.
- Le Papier de Recherche originel (Pour le côté académique)
  - "Efficient Capital Markets: A Review of Theory and Empirical Work" (Eugène Fama, 1970).
Pourquoi le lire : C'est le texte fondateur. C'est là qu'il définit les 3 formes d'efficience (Faible, Semi-Forte, Forte). Tu n'es pas obligé de lire les équations, mais lire l'introduction et la conclusion te donnera une vraie culture d'ingénieur financier. Tu peux le trouver gratuitement en PDF sur Google Scholar.
- Les Vidéos (YouTube)
  - La conférence du Prix Nobel (Nobel Prize Lecture) : Tape "Eugene Fama Nobel Lecture 2013" sur YouTube. C'est lui-même qui explique l'évolution de sa pensée sur 50 ans de carrière. C'est passionnant de voir un génie expliquer simplement sa théorie.
  - Les cours d'Aswath Damodaran (NYU Stern) : Tape "Damodaran Market Efficiency". Damodaran est le "Doyen de l'évaluation financière" à New York. Il a des vidéos de cours (en anglais) très pédagogiques où il explique la différence entre la théorie de Fama et la réalité des marchés.